In [1]:
import torch
from torch import nn
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments, BartConfig
from tqdm.notebook import tqdm
import pandas as pd
from datasets import Dataset, load_metric, DatasetDict
from torch.nn.init import xavier_uniform_
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
'''tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
print ("device ",device)
model = model.to(device)'''
def init_params(model):
    for name, param in model.named_parameters():
        if param.data.dim() > 1:
            xavier_uniform_(param.data)
        else:
            pass
model_name = 'facebook/bart-base'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
'''tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
print ("device ",device)
model = model.to(device)'''
mapping = {"N":"negation", "E":"entity swap", "#":"number swap", 
           "A":"to antonym", "I":"no information", 
           "X":"mutual exclusion"}
mapping2 = {"N":0, "E":1, "#":2, 
           "A":3, "I":4, 
           "X":5, "Ans":6}
mapping3 = {0:"N", 1:"E", 2:"#", 
           3:"A", 4:"I", 
           5:"X", 6:"Ans"}  

#model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)
config = BartConfig.from_pretrained(model_name)


class MTLModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Define a classification head
        self.classifier = nn.Linear(config.d_model, len(mapping2))
        self.dropout = nn.Dropout(0.2)
        self.bart = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
        self.criterion = nn.CrossEntropyLoss()
        self.loss_weights = [1, 0.2]
        init_params(self.classifier)
        self.optimizer = None
        self.scheduler = None

    def forward(self, input_ids, attention_mask, labels=None, 
                classification_input_ids=None, classification_attention_mask=None, classification_labels=None):
        # Generation outputs
        outputs = self.bart(input_ids=input_ids, attention_mask=attention_mask, labels=labels, output_hidden_states=True)
        
        # Classification outputs
        classification_outputs = self.bart(
            input_ids=classification_input_ids,
            attention_mask=classification_attention_mask,
            output_hidden_states=True
        )

        # Clone hidden states to avoid shared memory issues
        hidden_states = classification_outputs.decoder_hidden_states[-1].detach().clone()

        # Identify the EOS token
        eos_token_id = self.bart.config.eos_token_id
        eos_mask = classification_input_ids.eq(eos_token_id).detach()  # Ensure the mask is detached

        # Get the EOS hidden state
        eos_hidden_state = hidden_states[eos_mask].view(hidden_states.size(0), -1, hidden_states.size(-1))[:, -1, :].clone()

        eos_hidden_state = self.dropout(eos_hidden_state)
        
        # Apply classification head on the EOS hidden state
        classification_logits = self.classifier(eos_hidden_state)

        if self.training:
            #### Ensure classification_labels has the correct shape
            if classification_labels.dim() > 1:
                classification_labels = classification_labels.squeeze(-1).clone()  # Clone to ensure it's a separate tensor
            #### NEW
            # Convert classification_logits and classification_labels to the same floating point type
            classification_labels = classification_labels.long()
            
            # Calculate generation and classification loss
            loss_g = outputs.loss
            loss_r = self.criterion(classification_logits, classification_labels)
            ### NEW
            loss_r = loss_r.float()  # Ensure the loss is a floating-point type
            weighted_loss_g = self.loss_weights[0] * loss_g
            weighted_loss_r = self.loss_weights[1] * loss_r
            
            return weighted_loss_g + weighted_loss_r, classification_logits, loss_g

        # Modify input for generation using the classification result
        predicted_class = classification_logits.argmax(dim=-1).detach()  # Detach to ensure it does not affect gradient calculation
        modified_decoder_input_ids = self.modify_decoder_input(input_ids, predicted_class)

        # Generate text using modified input
        generation_outputs = self.bart.generate(input_ids=modified_decoder_input_ids, attention_mask=attention_mask)

        return generation_outputs, classification_logits

    def modify_decoder_input(self, input_ids, predicted_class):
        # Modify the input_ids based on the predicted class
        predicted_class_ids = predicted_class.unsqueeze(1).clone()  # Clone to ensure it's a separate tensor
        modified_input_ids = torch.cat([predicted_class_ids, input_ids[:, 1:].clone()], dim=1)  # Clone slices
        return modified_input_ids
    
    def configure_optimizers(self, learning_rate=5e-5, weight_decay=0.01, adam_epsilon=1e-8, warmup_steps=0, total_steps=0):
        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_grouped_parameters = [
            {
                "params": [
                    p for n, p in self.named_parameters()
                    if not any(nd in n for nd in no_decay)
                ],
                "weight_decay": weight_decay
            },
            {
                "params": [
                    p for n, p in self.named_parameters()
                    if any(nd in n for nd in no_decay)
                ],
                "weight_decay": 0.0
            }
        ]
        optimizer = AdamW(
            optimizer_grouped_parameters,
            lr=learning_rate,
            eps=adam_epsilon
        )
        self.optimizer = optimizer
        
        # Create the learning rate scheduler
        if warmup_steps > 0 and total_steps > 0:
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )
        else:
            scheduler = None
        self.scheduler = scheduler
        
        return optimizer, scheduler

    def optimizer_step(self, epoch, batch_idx):
        if self.optimizer is not None:
            self.optimizer.step()
            self.optimizer.zero_grad()
            if self.scheduler is not None:
                self.scheduler.step()

2024-08-21 12:32:27.537405: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-21 12:32:27.573553: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-21 12:32:28.326792: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(

In [2]:
from transformers import BartForConditionalGeneration, BartConfig, BartTokenizer
from safetensors.torch import load_file
import os

#output_dir = './checkpoint-41000'
output_dir = './fine-tuned-bartMTL-salience'

# Load the configuration
config = BartConfig.from_pretrained(output_dir)

# Reconstruct the model
model = MTLModel(config)
model.bart.model.encoder.load_state_dict(load_file(os.path.join(output_dir, "encoder.safetensors")))
model.bart.model.decoder.load_state_dict(load_file(os.path.join(output_dir, "decoder.safetensors")))
model.bart.model.shared.load_state_dict(load_file(os.path.join(output_dir, "shared.safetensors")))
model.bart.lm_head.load_state_dict(load_file(os.path.join(output_dir, "lm_head.safetensors")))
model.classifier.load_state_dict(load_file(os.path.join(output_dir, "classifier.safetensors")))

# Load the tokenizer
tokenizer = BartTokenizer.from_pretrained(output_dir)

# Now you can use `model` and `tokenizer` as needed
model.to(device)

MTLModel(
  (classifier): Linear(in_features=768, out_features=7, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (bart): BartForConditionalGeneration(
    (model): BartModel(
      (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (encoder): BartEncoder(
        (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
        (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
        (layers): ModuleList(
          (0-5): 6 x BartEncoderLayer(
            (self_attn): BartAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (activation_fn): GELUActivation()
      

In [5]:
# Example test case
context = "Example context, a long way farther, etc"
question = "What"
sal = "Example salient sentence"

def mission(context, question, label, sal):
    if label != "Ans":
        st = f"""Context: {context}

        Question: {question}

        Issue: {mapping[label]}

        Salient Sentence: {sal}

        Task: Generate a new question based on the context and salient sentence that can be answered with the given context, given the issue."""
    else:
        st = question
    return st

def labelLess(context, question, sal):
    st = f"""Context: {context}

    Question: {question}

    Salient Sentence: {sal}
    """
    return st

def evaluate_model(context, question, sal):
    # Move model to the correct device
    model.to(device)
    
    # Construct the input string for classification
    input_str = labelLess(context, question, sal)

    # Tokenize input for classification
    classification_inputs = tokenizer(input_str, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
    
    # Move input tensors to the same device as the model
    classification_inputs = {key: val.to(device) for key, val in classification_inputs.items()}
    
    # Predict the classification label
    with torch.no_grad():
        model.eval()
        outputs = model(
            input_ids=classification_inputs['input_ids'],
            attention_mask=classification_inputs['attention_mask'],
            classification_input_ids=classification_inputs['input_ids'],  # Not used in this step
            classification_attention_mask=classification_inputs['attention_mask'],  # Assuming attention_mask should be used
            classification_labels=None  # No labels needed for inference
        )
        
        # Extract classification logits
        classification_logits = outputs[1] if isinstance(outputs, tuple) else outputs['logits']
        predicted_label = classification_logits.argmax(dim=-1).item()
    
    # Prepare the mission input based on predicted label
    label_str = mapping3[predicted_label]
    mission_input = mission(context, question, label_str, sal)
    
    # Tokenize input for generation
    generation_inputs = tokenizer(mission_input, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
    
    # Move generation tensors to the same device as the model
    generation_inputs = {key: val.to(device) for key, val in generation_inputs.items()}
    
    # Generate text
    with torch.no_grad():
        model.eval()
        generated_ids = model(
            input_ids=generation_inputs['input_ids'],
            attention_mask=generation_inputs['attention_mask'],
            classification_input_ids=classification_inputs['input_ids'],
            classification_attention_mask=classification_inputs['attention_mask'],
            classification_labels=torch.tensor([-1] * generation_inputs['input_ids'].size(0)).to(device)  # Dummy labels
        )
        generated_ids = generated_ids[0] if isinstance(generated_ids, tuple) else generated_ids
        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    
    return mapping3[predicted_label], generated_text



print(evaluate_model(context, question, sal))


('A', 'What is another phrase meaning a long way farther, etc?')


In [6]:
df_test= pd.read_csv('dev_sal.csv')

def write(res_eval, fname):
    with open(fname, 'w') as f:
        for line in res_eval:
            f.write(f"{line}\n")
        f.close()
        
input_seq_eval = []
classes = []
# labels_seq_eval = []
for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    #label = row['label']
    context = row['context']
    question = row['original_question']
    sal = row['answer_sentence']
    class_s, input_s = evaluate_model(context, question, sal)
    classes.append(class_s)
    input_seq_eval.append(input_s)
    write(input_seq_eval, 'bart_epoch3.txt')
    write(classes, 'bart_epoch3_labels.txt')
    #break

df_test["BART_ANS"] = input_seq_eval
df_test["BART_LABEL"] = classes
df_test.to_csv("after_3_epochs.csv")
df_test.to_csv("MRC Eval/after_3_epochs.csv")

  0%|          | 0/5945 [00:00<?, ?it/s]

/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [7]:
!mv bart_epoch3_labels.txt "MRC Eval/"
!mv bart_epoch3.txt "MRC Eval/"

In [8]:
df_test.to_csv("MRC Eval/SG-Net/after_3_epochs.csv")

In [9]:
df_test

,Unnamed: 0.1,Unnamed: 0,qid,context,original_question,label,answer span,original_answer_span,answer_sentence,BART_ANS,BART_LABEL
0,5,5,5ad39d53604f3c001a3fe8d1,The Normans (Norman: Nourmands; French: Norman...,Who gave their name to Normandy in the 1000's ...,I,"('', '')",Normans,The Normans (Norman: Nourmands; French: Norman...,Who gave their name to Normandy in the 10th an...,A
1,6,6,5ad39d53604f3c001a3fe8d2,The Normans (Norman: Nourmands; French: Norman...,What is France a region of?,I,"('', '')",Normandy,The Normans (Norman: Nourmands; French: Norman...,What is Normandy a region of?,A
2,7,7,5ad39d53604f3c001a3fe8d3,The Normans (Norman: Nourmands; French: Norman...,Who did King Charles III swear fealty to?,I,"('', '')",Rollo,"They were descended from Norse (""Norman"" comes...",Who did King Charles III swear fealty to?,A
3,8,8,5ad39d53604f3c001a3fe8d4,The Normans (Norman: Nourmands; French: Norman...,When did the Frankish identity emerge?,E,"('', '')",10th century,The distinct cultural and ethnic identity of t...,When did the Normans identity emerge?,A
4,12,12,5ad3a266604f3c001a3fea27,"The Norman dynasty had a major political, cult...",What type of major impact did the Norman dynas...,X,"('', '')","political, cultural and military","The Norman dynasty had a major political, cult...",What type of major impact did the Norman dynas...,E
...,...,...,...,...,...,...,...,...,...,...,...
5940,11863,11863,5ad28a57d7d075001a4299b3,The connection between macroscopic nonconserva...,What does not change macroscopic closed systems?,N,"('', '')",nonconservative forces,The connection between macroscopic nonconserva...,What does change macroscopic closed systems?,A
5941,11869,11869,5ad28ad0d7d075001a4299cc,"The pound-force has a metric counterpart, less...",What does not have a metric counterpart?,N,"('', '')",pound-force,"The pound-force has a metric counterpart, less...",What is the metric counterpart of the pound-fo...,I
5942,11870,11870,5ad28ad0d7d075001a4299cd,"The pound-force has a metric counterpart, less...",What is the force exerted by standard gravity ...,E,"('', '')",kilogram-force,"The pound-force has a metric counterpart, less...",What is the pound-force primarily used for?,I
5943,11871,11871,5ad28ad0d7d075001a4299ce,"The pound-force has a metric counterpart, less...",What force leads to a commonly used unit of mass?,E,"('', '')",kilogram-force,"The pound-force has a metric counterpart, less...",What is the metric counterpart of the pound-fo...,I
